In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

In [5]:
df = pd.read_csv("features_output-all.csv")
print(df.shape)
df.head()

(582, 25)


,Orig_index,Name,Average Strength,Utract,U%_10_U_tract,U%_6_U_tract,A%_6_U_tract,C%_U_tract,SC_U_tract,Entropy_U_tract,...,Loop,Loop_len,GC%_Loop,Hairpin,Structure,Stem,Hairpin_entropy,Hairpin_len,Hairpin_SC,Hairpin_GC_count
0,0,ECK120029600,378.377966,UUUUCUUUUCUC,0.8,0.8333,0.0,0.2500,0.4545,0.4056,...,GUAA,0.0769,0.25,CCGCCGGUCUUGUCCACUACCUUGCAGUAAUGCGGUGGACAGGAUC...,((((((((((((((((((.....(((....))))))))))))))))...,CCGCCGGUCUUGUCCACUGCAUGCGGUGGACAGGAUCGGCGG,0.9376,42,31,7
1,1,ECK120033737,312.496371,UUUUUUUUUCGA,0.9,1.0000,0.0,0.0833,0.2727,0.6038,...,CUGACA,0.2308,0.50,GCCCGCACCUGACAGUGCGGGC,((((((((......)))))))),GCCCGCACGUGCGGGC,0.7718,16,11,6
2,2,pheA-1,243.526819,UUUUUUCGUUUU,0.8,1.0000,0.0,0.0833,0.2727,0.4083,...,GAAA,0.0769,0.25,GACGCUGAAAAGCGUC,((((((....)))))),GACGCUAGCGUC,0.9591,12,11,1
3,3,ECK120034435,239.913281,UUUUUUAUUGAU,0.8,1.0000,0.0,0.0000,0.4545,0.5204,...,AAAUC,0.1538,0.20,GGCCUCCCAAAUCGGGGGGCC,((((((((.....)))))))),GGCCUCCCGGGGGGCC,0.6359,16,7,4
4,4,ECK120033736,164.602167,UUUUUUAUUGCG,0.8,1.0000,0.0,0.0833,0.4545,0.7091,...,AUCAC,0.1538,0.40,GCCCCCGGAAGAUCACCUUCCGGGGGC,(((((((((((.....))))))))))),GCCCCCGGAAGCUUCCGGGGGC,0.8420,22,9,8


In [7]:
feature_cols = [col for col in df.columns if col not in ['Name', 'Average Strength', 'Orig_index', 'Utract', 'Atract', 'Stem', 'Hairpin', 'Loop', 'Structure']]
X = df[feature_cols]
y = df['Average Strength']
print(f"X shape: {X.shape}, y shape: {y.shape}")

#okay I need to remove the stuff the sequence stuff in the csv generation

X shape: (582, 16), y shape: (582,)


In [8]:
print(feature_cols)

['U%_10_U_tract', 'U%_6_U_tract', 'A%_6_U_tract', 'C%_U_tract', 'SC_U_tract', 'Entropy_U_tract', 'Entropy_A_tract', 'SC_A_tract', 'A%_6_A_tract', 'C%_6_A_tract', 'Loop_len', 'GC%_Loop', 'Hairpin_entropy', 'Hairpin_len', 'Hairpin_SC', 'Hairpin_GC_count']


In [16]:
model = XGBRegressor(random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
print(f"5-fold CV R²: {scores}")
print(f"Mean R²: {scores.mean():.4f} ± {scores.std():.4f}")

5-fold CV R²: [0.81899218 0.43620964 0.70089662 0.44793994 0.42225977]
Mean R²: 0.5653 ± 0.1635


In [ ]:
# best Hyperparameters combination from the preprint
model_tuned = XGBRegressor(
    n_estimators=4000,
    learning_rate=0.001,
    max_depth=5,
    min_child_weight=3,
    subsample=0.65,
    random_state=42 #the literature did 1996 for both xgboost and kfold, but I will just use 42 for both to be consistent with the rest of the code
)

scores_tuned = cross_val_score(model_tuned, X, y, cv=kf, scoring='r2')
print(f"5-fold CV R²: {scores_tuned}")
print(f"Mean R²: {scores_tuned.mean():.4f} ± {scores_tuned.std():.4f}")

5-fold CV R²: [0.88672784 0.48519346 0.70807102 0.60461969 0.5095712 ]
Mean R²: 0.6388 ± 0.1468


In [ ]:
import xgboost                                            
print(xgboost.__version__)

#shit, this version is much much higher than the preprint used. They used 1.7.6, but anyway I don't know what does the xgboost team changed.

3.2.0
